In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
# Load data — expects joint_library_predictions.csv produced by predict_joint_library.py
df = pd.read_csv('joint_library_predictions.csv')

# Assign cell line of origin based on name prefix
def get_origin(name):
    name_upper = name.upper()
    if name_upper.startswith('K562'):
        return 'K562'
    elif name_upper.startswith('HEPG2'):
        return 'HepG2'
    elif name_upper.startswith('WTC11'):
        return 'WTC11'
    else:
        return 'Other'

df['origin'] = df['name'].apply(get_origin)
print(df['origin'].value_counts())

# Keep only sequences from the three main cell lines
df = df[df['origin'].isin(['K562', 'HepG2', 'WTC11'])].copy()
print(f'\nTotal sequences: {len(df)}')

In [ ]:
# Map each origin to its "home" prediction column
home_pred = {
    'K562': 'k562_pred',
    'HepG2': 'hepg2_pred',
    'WTC11': 'wtc11_pred',
}

# Compute fold change: each model's prediction minus the home model's prediction (in log2 space)
# fold_change = model_pred - home_pred  (log2 scale, so this is log2(fold change))
for model in ['k562_pred', 'hepg2_pred', 'wtc11_pred']:
    col_name = f'fc_{model}'
    df[col_name] = df.apply(lambda row: row[model] - row[home_pred[row['origin']]], axis=1)

df[['name', 'origin', 'k562_pred', 'hepg2_pred', 'wtc11_pred', 
    'fc_k562_pred', 'fc_hepg2_pred', 'fc_wtc11_pred']].head(10)

In [ ]:
# Strip/swarm-style plot
# X-axis: cell line of origin (K562, HepG2, WTC11)
# Each point: a sequence, y = fold change vs home model
# Colors: red = K562 model, green = HepG2 model, blue = WTC11 model

model_colors = {
    'fc_k562_pred': ('red', 'K562 model'),
    'fc_hepg2_pred': ('green', 'HepG2 model'),
    'fc_wtc11_pred': ('blue', 'WTC11 model'),
}

origins = ['K562', 'HepG2', 'WTC11']

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, origin in zip(axes, origins):
    sub = df[df['origin'] == origin]
    
    for i, (fc_col, (color, label)) in enumerate(model_colors.items()):
        x_jitter = np.random.normal(i, 0.12, size=len(sub))
        ax.scatter(x_jitter, sub[fc_col], alpha=0.15, s=3, color=color, label=label, rasterized=True)
    
    ax.set_title(f'{origin} sequences', fontsize=13)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['K562\nmodel', 'HepG2\nmodel', 'WTC11\nmodel'])
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_ylabel('log2(fold change) vs home model' if ax == axes[0] else '')

# Single legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', fontsize=10, markerscale=3)

fig.suptitle('Fold change of model predictions relative to home cell-line model (PyTorch)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fold_change_strip.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics: median fold change per origin x model
summary = []
for origin in origins:
    sub = df[df['origin'] == origin]
    for fc_col, (color, label) in model_colors.items():
        summary.append({
            'Seq origin': origin,
            'Model': label,
            'Median log2FC': sub[fc_col].median(),
            'Mean log2FC': sub[fc_col].mean(),
            'Std': sub[fc_col].std(),
        })

pd.DataFrame(summary)

In [ ]:
# Box plot version for cleaner summary
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

for ax, origin in zip(axes, origins):
    sub = df[df['origin'] == origin]
    data = [sub['fc_k562_pred'], sub['fc_hepg2_pred'], sub['fc_wtc11_pred']]
    
    bp = ax.boxplot(data, positions=[0, 1, 2], widths=0.5, patch_artist=True,
                    showfliers=False, medianprops=dict(color='black', linewidth=1.5))
    
    for patch, color in zip(bp['boxes'], ['red', 'green', 'blue']):
        patch.set_facecolor(mcolors.to_rgba(color, alpha=0.4))
        patch.set_edgecolor(color)
    
    ax.set_title(f'{origin} sequences', fontsize=13)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['K562\nmodel', 'HepG2\nmodel', 'WTC11\nmodel'])
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_ylabel('log2(fold change) vs home model' if ax == axes[0] else '')

fig.suptitle('Fold change of model predictions relative to home cell-line model (PyTorch)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fold_change_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# For each origin, get the fold change to each NON-origin model,
# then bin sequences into top 5%, middle 5% (closest to 0), and bottom 5%.
# Output CSV columns: name, sequence, origin, comparison_model, bin, log2_fold_change

# Map origin -> list of non-origin fc columns
non_origin_map = {
    'K562':  [('fc_hepg2_pred', 'HepG2'), ('fc_wtc11_pred', 'WTC11')],
    'HepG2': [('fc_k562_pred', 'K562'),   ('fc_wtc11_pred', 'WTC11')],
    'WTC11': [('fc_k562_pred', 'K562'),   ('fc_hepg2_pred', 'HepG2')],
}

binned_rows = []

for origin in ['K562', 'HepG2', 'WTC11']:
    sub = df[df['origin'] == origin].copy()
    
    for fc_col, comp_model in non_origin_map[origin]:
        fc_vals = sub[fc_col]
        
        # Top 5%: highest fold change (non-origin predicts much higher than home)
        top_thresh = fc_vals.quantile(0.95)
        top_mask = fc_vals >= top_thresh
        
        # Bottom 5%: lowest fold change (non-origin predicts much lower than home)
        bot_thresh = fc_vals.quantile(0.05)
        bot_mask = fc_vals <= bot_thresh
        
        # Middle 5%: closest to 0 fold change (models agree most)
        abs_fc = fc_vals.abs()
        mid_thresh = abs_fc.quantile(0.05)
        mid_mask = abs_fc <= mid_thresh
        
        for mask, bin_label in [(top_mask, 'top_5pct'), (mid_mask, 'mid_5pct'), (bot_mask, 'bottom_5pct')]:
            selected = sub[mask].copy()
            selected = selected[['name', 'sequence', 'origin']].copy()
            selected['comparison_model'] = comp_model
            selected['bin'] = bin_label
            selected['log2_fold_change'] = fc_vals[mask].values
            binned_rows.append(selected)

binned_df = pd.concat(binned_rows, ignore_index=True)

print(binned_df.groupby(['origin', 'comparison_model', 'bin']).size().unstack(fill_value=0))
print(f'\nTotal sequences in output: {len(binned_df)}')

binned_df.to_csv('fold_change_binned_sequences.csv', index=False)
print('\nSaved to fold_change_binned_sequences.csv')